In [ ]:
# Creston Getz 7/13/2026
# This is the main file for the Grazioso Salvare Dashboard.
# It is a Dash app that allows users to filter and view data from the Austin Animal Center database.
# Dash communicates to MongoDB using APIs/HTTP. app.py and api.py implment the backend of the app.


import dash
import dash_leaflet as dl
from dash import dcc, html, dash_table
import dash_auth
from dash.dependencies import Input, Output, State
import plotly.graph_objects as go
import base64

import os
from dotenv import load_dotenv

import matplotlib.pyplot as plt
import plotly.express as px
import pandas as pd
import requests

###################################
# Database Connection and API Calls
###################################
API_URL = "http://127.0.0.1:5001"
load_dotenv() #load env vars


# This will make a API call to the backend to get data from the MongoDB and store it in a dataframe.
# The rescure parameter is used to filter the animals by the rescue type for the company.
# If no rescue type is selected, it will return all animals or no filter.
def get_animals(rescue_type=None):
    if rescue_type and rescue_type != 'None':
        response = requests.get(f"{API_URL}/api/animals/filter", 
        params={'rescue_type': rescue_type},
        headers={'X-API-KEY': os.environ['API_KEY']})
    else:
        # if no rescue type then return all animals
        response = requests.get(f"{API_URL}/api/animals", headers={'X-API-KEY': os.environ['API_KEY']})
    
    response.raise_for_status()
    df = pd.DataFrame(response.json())
    return df.drop(columns=['_id'], errors='ignore')

# Loads data into df. Empty if error
try:
    df = get_animals()
except Exception as e:
    print(f"Error with API call: {e}")
    df = pd.DataFrame() # Return empty dataframe if API fails


#########################
# Dashboard Layout / View
#########################
app = dash.Dash(__name__)
app.server.secret_key = os.environ['FLASK_SECRET_KEY']

# This code was inspired by https://community.plotly.com/t/flask-authentication-with-dash-pages/62958 used 7/13/26 and examples from Claude Code.
# Becuase this file is in a jupyter notebook the app will not start if env password is not set.
# It prevnts the app from not starting due to missing env var and locks the app with a password and username.
try: 
    dash_auth.BasicAuth( app, {'admin': os.environ['DASH_PASSWORD']},
        public_routes=["/_alive_<token>"],
    )
except KeyError:
    print("Auth Error")

#Grazisos Salvare Logo
image_filename = 'resources/Grazioso Salvare Logo.png'
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

# The app.layout section creates the HTMl code for the dashbaord.
app.layout = html.Div([
    #Style and Create the Header with the logo on the left
    html.Div(
    style={'display': 'flex', 'alignItems': 'center', 'justifyContent': 'center'},
    children=[
        html.Img(
            src='data:image/png;base64,{}'.format(encoded_image.decode()), #adds and decodes logo
            alt="Small picture of a red dog. Which is Grazioso Salvare Logo next to title",
            style={'width': '100px', 'height': 'auto', 'marginRight': '20px'}
        ),
        html.Div(
            style={'textAlign': 'center'},
            children=[
            html.H1('Grazioso Salvare Dashboard', style={'margin': '0'}),
            ]
        )
    ]),
    html.Hr(),
    # Section for the interactive filtering options. Adds a dropdown menu.
    html.Div([
        dcc.Dropdown(['None', 'Water Rescue', 'Mountain or Wilderness Rescue', 
                      'Disaster Rescue or Individual Tracking'], value = 'None',
                     id='filter-type', searchable=False, placeholder="Select a Filter",),
        html.P("Select a button on the left to update the geolocation map:", 
           style={'fontWeight': 'bold', 'color': '#000000', 'marginBottom': '5px'}),

        # outputs error message if there is a problem with dashboard.
        html.Div(id='error-message', style={'color': 'red', 'fontWeight': 'bold', 'marginTop': '10px'})

    ]),
    html.Hr(),
    # Create the main Datatable
    # dcc.Loading adds a loading circle to improve UI and UX if response is slow.
    # In testing all responses were very fast
    dcc.Loading(dash_table.DataTable(id='datatable-id',
                         columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
                         data=df.to_dict('records'),
        #features for interactive data table to make it user-friendly for client
        row_selectable = "single",
        selected_rows=[0],
        style_header={'fontWeight': 'bold',  #make header clearer
                        'text-transform': 'uppercase',
                      },
        
        style_data={'borderColor': '#333'},

        style_table={'overflowX': 'auto'}, #add scroll bar
        style_cell={'textAlign': 'left', #left algin text and add padding
                        'padding_right': '20px', 

                    }, 
        
        #enable pagination
        page_action='native',
        page_current=0,
        page_size=10,
        
        # add sorting and filtering to columns
        sort_action='native',
        sort_mode='multi',
        filter_action='native',
        filter_options={"placeholder_text": "Search column..."},

    ), type="circle"),
    
    html.Br(),
    html.Hr(),
    # This sets up the dashboard so pie chart and geolocation chart are side-by-side
    html.Div(className='row',
         style={'display' : 'flex', 'marginBottom':'50px'},
             children=[
        html.Div(
            id='graph-id',
            className='col s12 m6',

            ),
        html.Div(
            id='map-id',
            className='col s12 m6',
            )
        ])
])

#############################################
# Interaction Between Components / Controller
#############################################
# Updates the datatable and chart based on filter the user selects. Each time it is called it will call the get animals method.
@app.callback(
    [Output('datatable-id', 'data'),
    Output('datatable-id', 'selected_rows'),
    Output('error-message', 'children')],
    [Input('filter-type', 'value')]
)
# This method was inspired by 
# https://dash.plotly.com/callback-error-handlers accessed 7/14/26
# https://dash.plotly.com/basic-callbacks accessed 7/14/26
# And a few examples from Claude Code
# I was having trouble handling the errors when the user would select differnt filters. 
def update_dashboard(filter_type):
    try:
        records = get_animals(filter_type).to_dict('records')
        return records, [0] if records else [], '' # resets selection when the data is swapped. records is always returned as is
    except requests.exceptions.RequestException as e:
        return [], [], html.Div(
            "Sorry the dashboard could not be loaded. Please try again.", style={'color': 'red', 'fontWeight': 'bold'}
        )
    except Exception:
        return [], [], html.Div(
            "An error occurred loading the dashboard. Try refreshing the page.", style={'color': 'red', 'fontWeight': 'bold'}
        )



# Display the breeds of animal based on quantity represented in
# the data table. This method makes a pie chart based on whatever filter the user selects
# to show the distribution of dog breeds only
@app.callback(
     Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")])
def update_graphs(viewData):
    #return nothing is no data
    if viewData is None:
        return []
    
    #Creates a dataframe of only dogs and will return a p tag if no dog data is found
    dff = pd.DataFrame.from_dict(viewData)
    dff = dff[dff['animal_type'] == 'Dog']
    if dff.empty:
        return [html.P("No dog data available for this filter.")]
    
    #Gets the top 10 breed value counts for dogs
    top_breeds = dff['breed'].value_counts().nlargest(10).reset_index()
    top_breeds.columns = ['breed', 'count']
    
    #returns figure
    return [
       dcc.Graph(            
           figure = px.pie(top_breeds, values='count', names='breed', title='Top 10 Dog Breed Distribution')
       )
    ]

    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    if selected_columns is None:
        return []
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
# TODO: Use loc instead of iloc
# TODO: This callback is not yet completed. It will be finished after the database is done.
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):
    # if viewdata or index is none then the callback fired before the table was populated. 
    if viewData is None:
        return
    elif index is None:
        return
    if not viewData or not index: 
        return [html.P("Select a row in the table to view on the map")]
    
    dff = pd.DataFrame.from_dict(viewData) # convert view data to a dataframe
    row = index[0]

    if row >= len(dff):
        return [html.P("Select a row in the table to view on the map")]

        
    # Austin TX is at [30.75,-97.48]
    # TODO: The order and schema will need to match the data that is outputted. When the real database is created this will need to be updated
    return [
        dl.Map(style={'width': '1000px', 'height': '500px'}, center=[30.75,-97.48], zoom=10, children=[
            dl.TileLayer(id="base-layer-id"),
            # Marker with tool tip and popup
            # Column 13 and 14 define the grid-coordinates for the map
            # Column 4 defines the breed for the animal
            # Column 9 defines the name of the animal
            
            # Marker will display the GPS coords upon hover, along with other info about
            # the animal in a popup menu upon clicking
            dl.Marker(position=[dff.loc[row,'location_lat'],dff.loc[row,'location_long']], children=[
                dl.Tooltip(f"Lat: {dff.loc[row,'location_lat']:.4} Long: {dff.loc[row,'location_long']:.4}"),
                #dl.Popup([
                #html.H1("Animal Info"),
                #html.P(f"Name: {dff.iloc[row,9]}"),
                #html.P(f"Type: {dff.iloc[row,3]}"),
                #html.P(f"Breed: {dff.iloc[row,4]}"),
               #html.P(f"ID: {dff.iloc[row,2]}"),
               # ])
            ])
        ])
    ]


# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run(debug=True, port=8050, jupyter_mode='inline', mode='external') 

 http://127.0.0.1:8050